# Stim in the QDK

`qdk.stim` compiles [Stim](https://github.com/quantumlib/Stim) circuits to QIR and simulates
them, with extensions for **qubit loss**, **post-selection**, and **non-Clifford gates**.

| Function | Returns |
| --- | --- |
| `stim.compile(src, noise=None)` | `(qir, noise)` |
| `stim.run(src, shots=1, noise=None, seed=None, type=None)` | one result list per shot |

Every measurement records `Zero`, `One`, or `Loss`, displayed below as `0`, `1`, and `L`.

`type` picks the simulator:

- `"clifford"` — stabilizer simulator. Scales to many qubits and absorbs a modest number of
  non-Clifford operations by branching the stabilizer decomposition.
- `"cpu"` / `"gpu"` — full state vector, for circuits dominated by non-Clifford gates.
- `None` (default) — `"gpu"` when a GPU is available, otherwise `"cpu"`.

> `qdk.stim` is experimental and its API may change.

In [ ]:
from qdk import stim
from qdk.widgets import Histogram

SHOTS = 2000

## Compiling to QIR

Everything in the [Stim gate reference](https://github.com/quantumlib/Stim/blob/main/doc/gates.md)
is supported, except for the handful of instructions listed at the end of this notebook, and
the QDK adds the extensions covered below.

`stim.compile` lowers a circuit to QIR and returns it alongside the noise configuration that
the simulators consume.

In [ ]:
bell = """H 0
CX 0 1
MR 0 1
"""

qir, _ = stim.compile(bell)
print(qir)

`stim.run` compiles and simulates in one step. The Bell pair is entangled, so only `00` and
`11` occur.

In [ ]:
Histogram(stim.run(bell, shots=SHOTS, type="clifford"), labels="kets")

## Noise channels

| Instruction | Effect |
| --- | --- |
| `X_ERROR(p)`, `Y_ERROR(p)`, `Z_ERROR(p)` | Independent Pauli error on each target |
| `DEPOLARIZE1(p)`, `DEPOLARIZE2(p)` | Uniform Pauli error per qubit or per pair |
| `PAULI_CHANNEL_1(...)`, `PAULI_CHANNEL_2(...)` | Explicit per-Pauli probabilities |
| `CORRELATED_ERROR(p)`, `E(p)` | One Pauli product applied as a single event |
| `ELSE_CORRELATED_ERROR(p)` | Another branch of the preceding correlated error |
| `LOSS_ERROR(p)` | Loses each target with probability $p$ |

Every Pauli target on a `CORRELATED_ERROR` line belongs to one event, so `X0 X1` fires on both
qubits or on neither. `ELSE_CORRELATED_ERROR` adds a branch that is reached only when no
earlier link in the chain fired, making the branches mutually exclusive.

In [ ]:
correlated = """CORRELATED_ERROR(0.2) X0 X1
ELSE_CORRELATED_ERROR(0.2) X0
MR 0 1
"""

Histogram(stim.run(correlated, shots=SHOTS, type="clifford"), labels="kets")

### Readout noise

Any instruction that appends to the measurement record takes an optional probability that
flips the recorded bit, leaving the qubit itself untouched: `M` / `MZ`, `MX`, `MY`, the
`MR` variants, the pair measurements `MXX` / `MYY` / `MZZ`, `MPP`, and `PEEK_LOSS`.

Below both qubits stay in $|0\rangle$, so every `1` in the histogram is a misread.

In [ ]:
readout_noise = """M(0.1) 0 1
"""

Histogram(stim.run(readout_noise, shots=SHOTS, type="clifford"), labels="kets")

### Loss

Qubit loss is a QDK extension. `LOSS_ERROR(p)` loses each target with probability $p$, and
measuring a lost qubit records `Loss` rather than a bit.

In [ ]:
loss = """LOSS_ERROR(0.15) 0 1
MR 0 1
"""

Histogram(stim.run(loss, shots=SHOTS, type="clifford"), labels="kets")

Loss also has a target form, `L0`,
which may be combined with Pauli terms inside a correlated error to build branches that mix
the two.

In [ ]:
mixed_loss = """CORRELATED_ERROR(0.1) L0
ELSE_CORRELATED_ERROR(0.1) L1
ELSE_CORRELATED_ERROR(0.1) L0 X1
MR 0 1
"""

Histogram(stim.run(mixed_loss, shots=SHOTS, type="clifford"), labels="kets")

### Inspecting loss with `PEEK_LOSS`

`PEEK_LOSS` reports whether each target is currently lost, appending `1` for a lost qubit and
`0` otherwise. It neither measures the qubit nor clears the loss, so a later measurement of a
lost qubit still records `Loss`.

In [ ]:
peek = """LOSS_ERROR(0.3) 0
PEEK_LOSS 0
M 0
"""

Histogram(stim.run(peek, shots=SHOTS, type="clifford"), labels="kets")

## Post-selection with `SELECT`

A `SELECT { ... }` block re-runs its own body until every condition inside it passes.

- `REQUIRE rec[...]` restarts the block unless the referenced records have even parity, so
  `REQUIRE rec[-1]` keeps only shots whose last measurement was `0`.
- A lost qubit has no bit to contribute to that parity, so `REQUIRE` also restarts whenever
  one of its records was lost
- Prefixing a record with `!` inverts it, so `REQUIRE !rec[-1]` keeps the shots that
  measured `1`.
- Conditions are checked where they appear, letting one block select several measurements in
  sequence.

In [ ]:
select = """SELECT {
    H 0
    M 0
    REQUIRE rec[-1]
    H 1
    M 1
    REQUIRE !rec[-1]
}
"""

Histogram(stim.run(select, shots=SHOTS, type="clifford"), labels="kets")

Listing several records in one `REQUIRE` selects on their joint parity instead of on each
record individually, which keeps the two qubits below correlated rather than fixed.

In [ ]:
parity = """SELECT {
    H 0
    H 1
    M 0
    M 1
    REQUIRE rec[-1] rec[-2]
}
"""

Histogram(stim.run(parity, shots=SHOTS, type="clifford"), labels="kets")

### Discarding lost qubits with `NOTLEAKED`

`NOTLEAKED rec[...]` is the loss half of `REQUIRE` on its own: it restarts the block when a
referenced measurement was lost, but places no constraint on the recorded bit. Use it when a
shot should survive with either outcome, just not with `L`.

It cannot be negated, and it cannot reference a `PEEK_LOSS` record — a peek succeeds even
when the qubit is lost, so the request would be ambiguous.

In [ ]:
not_leaked = """SELECT {
    H 0
    LOSS_ERROR(0.3) 0
    MR 0
    NOTLEAKED rec[-1]
}
"""

Histogram(stim.run(not_leaked, shots=SHOTS, type="clifford"), labels="kets")

### Nesting and record scope

Blocks nest, and a restart re-runs only the body of the block that failed. A record is *in
scope* if it was produced inside the current block or a nested one; records from an enclosing
block are out of scope because a restart can no longer change them.

Below, the inner block fixes qubit 0 and the outer block fixes qubit 1, so only `00` survives.

In [ ]:
nested = """SELECT {
    SELECT {
        H 0
        M 0
        REQUIRE rec[-1]
    }
    H 1
    M 1
    REQUIRE rec[-1]
}
"""

Histogram(stim.run(nested, shots=SHOTS, type="clifford"), labels="kets")

Every condition must reference at least one in-scope record, otherwise a restart could never
satisfy it and the block would loop forever; that case is rejected at compile time. Mixing an
out-of-scope record with an in-scope one is allowed, and the outer record then acts as a fixed
value to select against.

In [ ]:
scoping = """H 0
M 0
SELECT {
    H 1
    M 1
    REQUIRE rec[-1] rec[-2]
}
"""

Histogram(stim.run(scoping, shots=SHOTS, type="clifford"), labels="kets")

## `REPEAT` blocks

`REPEAT N { ... }` unrolls its body `N` times at compile time, and each iteration appends its
own measurement records. `N` must be greater than zero.

In [ ]:
repeat = """REPEAT 3 {
    X 0
    M 0
}
"""

Histogram(stim.run(repeat, shots=SHOTS, type="clifford"), labels="kets")

## Pauli products

`MPP` measures a Pauli product written with `*`, such as `X0*Y1*Z2`, and `SPP` / `SPP_DAG`
apply the generalized `S` gate $\exp(\mp i \frac{\pi}{4} P)$ to one. Multiple products may
share a line, separated by whitespace. The non-Clifford `TPP`, `TPP_DAG`, and `R_PAULI` take
the same targets and are covered further below.

- A `!` on any factor negates the whole product, so `MPP !Z0*Z1` and `MPP Z0*!Z1` agree, and
  `SPP !Z0` matches `SPP_DAG Z0`.
- Repeated factors on the same qubit are folded by Pauli multiplication, so `X0*Y1*Y1`
  reduces to `X0`.
- Folding can leave a factor of $\pm i$, which makes the product anti-Hermitian. Those
  products, such as `X0*Z0`, are rejected.

On a Bell pair both $Z_0Z_1$ and $X_0X_1$ measure `0` with certainty, and negating the first
product flips its outcome.

In [ ]:
pauli_measurement = """H 0
CX 0 1
MPP Z0*Z1 X0*X1 !Z0*Z1
"""

Histogram(stim.run(pauli_measurement, shots=SHOTS, type="clifford"), labels="kets")

`SPP Z` is exactly `S`. Applying it twice gives `Z`, which the surrounding Hadamards turn into
a bit flip, while `SPP Z` followed by `SPP !Z` cancels.

In [ ]:
generalized_s = """H 0 1
SPP Z0
SPP Z0
SPP Z1
SPP !Z1
H 0 1
M 0 1
"""

Histogram(stim.run(generalized_s, shots=SHOTS, type="clifford"), labels="kets")

## Non-Clifford extensions

The QDK extends Stim with the non-Clifford gates of
[Clifft](https://github.com/unitaryfoundation/clifft/blob/main/docs/reference/gates.md).

| Category | Instructions | Grouping |
| --- | --- | --- |
| Phase gates | `T`, `T_DAG` | one qubit each |
| | `TPP`, `TPP_DAG` | one Pauli product each |
| Controlled gates | `CH` | consecutive pairs |
| | `CCX`, `CCZ` | consecutive triples |
| Single-qubit rotations | `R_X(a)`, `R_Y(a)`, `R_Z(a)` | one qubit each |
| | `U3(t, p, l)`, `U(t, p, l)` | one qubit each |
| Pair rotations | `R_XX(a)`, `R_YY(a)`, `R_ZZ(a)` | consecutive pairs |
| Pauli rotation | `R_PAULI(a)` | one Pauli product each |

**Angles.** A bare argument counts half turns, so `R_X(0.5)` rotates by $\pi/2$. Append `rad`
to give radians directly, as in `R_X(0.5rad)`. `U3(theta, phi, lambda)` applies
$R_Z(\varphi) R_Y(\theta) R_Z(\lambda)$ and may mix the two units. `U` is an alias for `U3`.

**Simulation.** `type="clifford"` branches the stabilizer decomposition on every non-Clifford
operation, so it stays efficient while they remain sparse. Reach for `type="cpu"` or
`type="gpu"` when they do not.

`TPP Z` is exactly `T`, so qubits 0 and 1 both accumulate $T^4 = Z$ and end up flipped.
`TPP X` instead rotates about $X$, which leaves $|+\rangle$ alone, and the `!` on qubit 3
inverts the second gate so that the pair cancels.

In [ ]:
phase_gates = """H 0 1 2 3
REPEAT 4 {
    T 0
    TPP Z1
    TPP X2
}
T 3
TPP !Z3
H 0 1 2 3
M 0 1 2 3
"""

Histogram(stim.run(phase_gates, shots=SHOTS, type="clifford"), labels="kets")

`CCX` flips its target and `CCZ` applies a phase flip once both controls are `1`; the
Hadamards around `CCZ` expose that phase flip in the computational basis. `CH` applies a
Hadamard when its control is `1`, leaving qubit 4 in an even superposition.

In [ ]:
controlled_gates = """X 0 1
CCX 0 1 2
H 3
CCZ 0 1 3
H 3
CH 0 4
M 0 1 2 3 4
"""

Histogram(stim.run(controlled_gates, shots=SHOTS, type="clifford"), labels="kets")

`R_X(1)` is a half turn about $X$ and flips qubit 0. `R_Y(1rad)` rotates by a single radian,
so qubit 1 measures `1` with probability $\sin^2(1/2) \approx 0.23$. `U(1, 0, 0)` reduces to
$R_Y(\pi)$ and flips qubit 2.

In [ ]:
rotations = """R_X(1) 0
R_Y(1rad) 1
U(1, 0, 0) 2
M 0 1 2
"""

Histogram(stim.run(rotations, shots=SHOTS, type="clifford"), labels="kets")

`R_XX(1)` and `R_PAULI(1) X*X` are the same half turn about $X \otimes X$ and flip both of
their qubits, while `R_ZZ(0.5)` only adds a relative phase that the computational basis cannot
see.

In [ ]:
pauli_rotations = """R_ZZ(0.5) 0 1
R_XX(1) 2 3
R_PAULI(1) X4*X5
M 0 1 2 3 4 5
"""

Histogram(stim.run(pauli_rotations, shots=SHOTS, type="clifford"), labels="kets")

## Not yet supported

Tracked in [microsoft/qdk#3518](https://github.com/microsoft/qdk/issues/3518).

| Feature | Current behavior |
| --- | --- |
| `HERALDED_ERASE`, `HERALDED_PAULI_CHANNEL_1` | Compile error: unsupported instruction |
| Sweep-bit targets, such as `CX sweep[5] 7` | Compile error: unsupported target |
| Pauli products that fold to the identity, such as `MPP Z0*Z0` | Compile error: unsupported target |
| `DETECTOR`, `OBSERVABLE_INCLUDE`, `QUBIT_COORDS`, `SHIFT_COORDS`, `TICK`, `MPAD` | Parsed and ignored; there is no detector or observable sampling |
| Instruction tags, such as `H[tag] 0` | Parsed and ignored |